In [9]:
# !pip install numpy pandas matplotlib seaborn nltk tensorflow

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pandas as pd

In [ ]:
# Carregando os datasets de treino e validação
train_file_path = '/twitter_training.csv'
validation_file_path = '/twitter_validation.csv'

df_train = pd.read_csv(train_file_path)
df_validation = pd.read_csv(validation_file_path)

In [ ]:
df_train.columns = ['ID', 'Category', 'Sentiment', 'Tweet']
df_validation.columns = ['ID', 'Category', 'Sentiment', 'Tweet']

# Excluindo linhas com tweets ausentes em ambos os datasets
df_train.dropna(subset=['Tweet'], inplace=True)
df_validation.dropna(subset=['Tweet'], inplace=True)

In [ ]:
def analyze_dataset(df, dataset_name):
    print(f"Análise do dataset: {dataset_name}")

    # Exibindo informações básicas
    print(df.info())

    # Exibindo a contagem de sentimentos
    sentiment_counts = df['Sentiment'].value_counts()
    print(f"\nContagem de sentimentos para {dataset_name}:")
    print(sentiment_counts)

    # Visualizando a distribuição dos sentimentos
    plt.figure(figsize=(8, 6))
    sns.countplot(x='Sentiment', data=df, palette='viridis', order=sentiment_counts.index)
    plt.title(f'Distribuição das Classes de Sentimento ({dataset_name})')
    plt.ylabel('Contagem')
    plt.xlabel('Sentimento')
    plt.show()

    # Exibindo um exemplo de tweet para cada classe de sentimento
    example_tweets = df.groupby('Sentiment')['Tweet'].first()
    print(f"\nExemplos de tweets de cada classe para {dataset_name}:")
    print(example_tweets)

In [ ]:
# Realizando a análise exploratória para o dataset de treino
analyze_dataset(df_train, "Treino")


In [ ]:
# Realizando a análise exploratória para o dataset de validação
analyze_dataset(df_validation, "Validação")

In [ ]:
class MetricsCallback:
    def __init__(self):
        self.train_accuracy = []
        self.val_accuracy = []
        self.train_f1 = []
        self.val_f1 = []
        self.train_precision = []
        self.val_precision = []
        self.train_recall = []
        self.val_recall = []
        self.train_loss = []
        self.val_loss = []

    def update_metrics(self, epoch, y_true_train, y_pred_train, y_true_val, y_pred_val, train_loss, val_loss):
        y_pred_train = torch.argmax(y_pred_train, dim=1).cpu().numpy()
        y_pred_val = torch.argmax(y_pred_val, dim=1).cpu().numpy()

        # Calculando as métricas para o conjunto de treino
        train_acc = (y_true_train == y_pred_train).mean()
        train_f1 = f1_score(y_true_train, y_pred_train, average='weighted')
        train_precision = precision_score(y_true_train, y_pred_train, average='weighted')
        train_recall = recall_score(y_true_train, y_pred_train, average='weighted')

        # Calculando as métricas para o conjunto de validação
        val_acc = (y_true_val == y_pred_val).mean()
        val_f1 = f1_score(y_true_val, y_pred_val, average='weighted')
        val_precision = precision_score(y_true_val, y_pred_val, average='weighted')
        val_recall = recall_score(y_true_val, y_pred_val, average='weighted')

        # Armazenando as métricas e a perda
        self.train_accuracy.append(train_acc)
        self.val_accuracy.append(val_acc)
        self.train_f1.append(train_f1)
        self.val_f1.append(val_f1)
        self.train_precision.append(train_precision)
        self.val_precision.append(val_precision)
        self.train_recall.append(train_recall)
        self.val_recall.append(val_recall)
        self.train_loss.append(train_loss)
        self.val_loss.append(val_loss)

        # Exibindo as métricas no final de cada época
        print(f"Epoch {epoch + 1} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        print(f"Train Acc: {train_acc:.4f}, Train F1: {train_f1:.4f}, Train Precision: {train_precision:.4f}, Train Recall: {train_recall:.4f}")
        print(f"Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}, Val Precision: {val_precision:.4f}, Val Recall: {val_recall:.4f}")
        

In [ ]:
def plot_metrics(metrics_callback):
    epochs = range(1, len(metrics_callback.train_accuracy) + 1)

    plt.figure(figsize=(12, 10))

    # Plotando a acurácia
    plt.subplot(2, 3, 1)
    plt.plot(epochs, metrics_callback.train_accuracy, label='Train Accuracy')
    plt.plot(epochs, metrics_callback.val_accuracy, label='Val Accuracy')
    plt.title('Accuracy per Epoch')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # Plotando o F1-Score
    plt.subplot(2, 3, 2)
    plt.plot(epochs, metrics_callback.train_f1, label='Train F1-Score')
    plt.plot(epochs, metrics_callback.val_f1, label='Val F1-Score')
    plt.title('F1-Score per Epoch')
    plt.xlabel('Epochs')
    plt.ylabel('F1-Score')
    plt.legend()

    # Plotando a precisão
    plt.subplot(2, 3, 3)
    plt.plot(epochs, metrics_callback.train_precision, label='Train Precision')
    plt.plot(epochs, metrics_callback.val_precision, label='Val Precision')
    plt.title('Precision per Epoch')
    plt.xlabel('Epochs')
    plt.ylabel('Precision')
    plt.legend()

    # Plotando o recall
    plt.subplot(2, 3, 4)
    plt.plot(epochs, metrics_callback.train_recall, label='Train Recall')
    plt.plot(epochs, metrics_callback.val_recall, label='Val Recall')
    plt.title('Recall per Epoch')
    plt.xlabel('Epochs')
    plt.ylabel('Recall')
    plt.legend()

    # Plotando a perda (loss)
    plt.subplot(2, 3, 5)
    plt.plot(epochs, metrics_callback.train_loss, label='Train Loss')
    plt.plot(epochs, metrics_callback.val_loss, label='Val Loss')
    plt.title('Loss per Epoch')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=5, patience=3, min_delta=0.001):
    metrics_callback = MetricsCallback()
    early_stopping = EarlyStopping(patience=patience, min_delta=min_delta)
    model.to(device)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        all_train_preds = []
        all_train_labels = []

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            all_train_preds.append(outputs.detach())
            all_train_labels.append(labels.detach())

        model.eval()
        val_loss = 0.0
        all_val_preds = []
        all_val_labels = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                all_val_preds.append(outputs)
                all_val_labels.append(labels)

        all_train_preds = torch.cat(all_train_preds)
        all_train_labels = torch.cat(all_train_labels)
        all_val_preds = torch.cat(all_val_preds)
        all_val_labels = torch.cat(all_val_labels)

        # Atualizando as métricas ao final de cada época
        metrics_callback.update_metrics(
            epoch,
            all_train_labels.cpu().numpy(),
            all_train_preds.cpu(),
            all_val_labels.cpu().numpy(),
            all_val_preds.cpu(),
            train_loss / len(train_loader),
            val_loss / len(val_loader)
        )

        # Verificar Early Stopping com base no val_loss
        early_stopping(val_loss / len(val_loader))
        if early_stopping.early_stop:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    # Retornando as métricas ao final do treinamento para plotar
    return metrics_callback


In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):

        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        # Se for a primeira época, define o melhor loss
        if self.best_loss is None:
            self.best_loss = val_loss
        # Verifica se a melhora em val_loss é significativa
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        # Se o val_loss melhorar, reseta o contador
        else:
            self.best_loss = val_loss
            self.counter = 0


In [ ]:
def clean_tweet(tweet):
    # Convertendo para minúsculas
    tweet = tweet.lower()

    # Removendo URLs
    tweet = re.sub(r'http\S+|www\S+|https\S+', '', tweet, flags=re.MULTILINE)

    # Removendo menções (@usuário)
    tweet = re.sub(r'@\w+', '', tweet)

    # Removendo caracteres especiais, exceto espaços, letras e números
    tweet = re.sub(r'[^A-Za-z0-9\s]', '', tweet)

    # Removendo múltiplos espaços
    tweet = re.sub(r'\s+', ' ', tweet).strip()


    return tweet

In [ ]:
df_train['Cleaned_Tweet'] = df_train['Tweet'].apply(clean_tweet)
df_validation['Cleaned_Tweet'] = df_validation['Tweet'].apply(clean_tweet)

In [ ]:
df_train.drop_duplicates(subset=['Cleaned_Tweet'], inplace=True)
df_validation.drop_duplicates(subset=['Cleaned_Tweet'], inplace=True)

In [ ]:
print("\nExemplo de tweet original e limpo (Treino):")
print(f"Original: {df_train['Tweet'].iloc[0]}")
print(f"Limpo: {df_train['Cleaned_Tweet'].iloc[0]}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
X_train = df_train.iloc[:, 3].fillna('').astype(str).values  # Tratar valores nulos e converter para string
y_train = df_train.iloc[:, 2].values  # 3ª coluna para sentimento

X_val = df_validation.iloc[:, 3].fillna('').astype(str).values  # Tratar valores nulos e converter para string
y_val = df_validation.iloc[:, 2].values  # 3ª coluna para sentimento

In [ ]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_val = label_encoder.transform(y_val)


In [ ]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

In [ ]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)


In [ ]:
# Padronizando o comprimento das sequências
max_len = 50
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='post')

In [ ]:
train_dataset = TensorDataset(torch.tensor(X_train_pad, dtype=torch.long), torch.tensor(y_train, dtype=torch.long))
val_dataset = TensorDataset(torch.tensor(X_val_pad, dtype=torch.long), torch.tensor(y_val, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, vocab_size, embedding_dim):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        lstm_out = lstm_out[:, -1, :]  # Pegando a última saída da LSTM
        out = self.fc(lstm_out)
        return out

In [ ]:
vocab_size = 10000
embedding_dim = 64
hidden_dim = 128
output_dim = 4

In [ ]:
model = LSTMClassifier(input_dim=embedding_dim, hidden_dim=hidden_dim, output_dim=output_dim, vocab_size=vocab_size, embedding_dim=embedding_dim)
model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
metrics_callback = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=20, patience=3, min_delta=0.001)


In [ ]:
plot_metrics(metrics_callback)


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
model.eval()
all_val_preds = []
all_val_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        all_val_preds.append(outputs)
        all_val_labels.append(labels)


In [ ]:
all_val_preds = torch.cat(all_val_preds)
all_val_labels = torch.cat(all_val_labels)

# Convertendo as previsões para rótulos
y_pred_val = torch.argmax(all_val_preds, dim=1).cpu().numpy()
y_true_val = all_val_labels.cpu().numpy()

# Calculando a matriz de confusão
cm = confusion_matrix(y_true_val, y_pred_val)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Matriz de Confusão')
plt.show()

In [ ]:
# Definindo o modelo GRU
class GRUClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, vocab_size, embedding_dim):
        super(GRUClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        gru_out, _ = self.gru(embedded)
        gru_out = gru_out[:, -1, :]  # Pegando a última saída da GRU
        out = self.fc(gru_out)
        return out


In [ ]:
model = GRUClassifier(input_dim=embedding_dim, hidden_dim=hidden_dim, output_dim=output_dim, vocab_size=vocab_size, embedding_dim=embedding_dim)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
metrics_callback = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=20, patience=3, min_delta=0.001)


In [ ]:
plot_metrics(metrics_callback)


In [ ]:
model.eval()
all_val_preds = []
all_val_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        all_val_preds.append(outputs)
        all_val_labels.append(labels)


In [ ]:
all_val_preds = torch.cat(all_val_preds)
all_val_labels = torch.cat(all_val_labels)

# Convertendo as previsões para rótulos
y_pred_val = torch.argmax(all_val_preds, dim=1).cpu().numpy()
y_true_val = all_val_labels.cpu().numpy()

# Calculando a matriz de confusão
cm = confusion_matrix(y_true_val, y_pred_val)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Matriz de Confusão')
plt.show()

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, vocab_size, embedding_dim):
        super(BiLSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.bilstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)  # Multiplica por 2 por ser bidirecional

    def forward(self, x):
        embedded = self.embedding(x)
        bilstm_out, _ = self.bilstm(embedded)
        bilstm_out = bilstm_out[:, -1, :]  # Pegando a última saída da Bi-LSTM
        out = self.fc(bilstm_out)
        return out

In [ ]:
model = BiLSTMClassifier(input_dim=embedding_dim, hidden_dim=hidden_dim, output_dim=output_dim, vocab_size=vocab_size, embedding_dim=embedding_dim)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
metrics_callback = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=20, patience=3, min_delta=0.001)


In [ ]:
plot_metrics(metrics_callback)


In [ ]:
model.eval()
all_val_preds = []
all_val_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        all_val_preds.append(outputs)
        all_val_labels.append(labels)

In [ ]:
all_val_preds = torch.cat(all_val_preds)
all_val_labels = torch.cat(all_val_labels)

# Convertendo as previsões para rótulos
y_pred_val = torch.argmax(all_val_preds, dim=1).cpu().numpy()
y_true_val = all_val_labels.cpu().numpy()

# Calculando a matriz de confusão
cm = confusion_matrix(y_true_val, y_pred_val)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Matriz de Confusão')
plt.show()

In [ ]:
import torch.nn.functional as F


In [ ]:
class CNNTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, output_dim, kernel_sizes, num_filters):
        super(CNNTextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # Camadas de convolução com diferentes tamanhos de filtro
        self.conv_layers = nn.ModuleList([
            nn.Conv2d(in_channels=1, out_channels=num_filters, kernel_size=(k, embedding_dim)) for k in kernel_sizes
        ])

        # Camada densa final
        self.fc = nn.Linear(len(kernel_sizes) * num_filters, output_dim)

    def forward(self, x):
        x = self.embedding(x).unsqueeze(1)  # Adiciona uma dimensão de canal (batch, 1, seq_len, embed_dim)

        # Aplicando convoluções e ReLU
        conved = [F.relu(conv(x)).squeeze(3) for conv in self.conv_layers]

        # Max-pooling para reduzir a dimensionalidade
        pooled = [F.max_pool1d(conv, conv.shape[2]).squeeze(2) for conv in conved]

        # Concatenando as saídas das diferentes convoluções
        cat = torch.cat(pooled, dim=1)

        # Passando pela camada densa final
        return self.fc(cat)

In [ ]:
vocab_size = 10000
embedding_dim = 64
output_dim = 4  # Número de classes (positivo, negativo, neutro, irrelevante)
kernel_sizes = [3, 4, 5]  # Tamanhos dos filtros (n-gramas de 3, 4 e 5 palavras)
num_filters = 100  # Número de filtros por tamanho de kernel

In [ ]:
model = CNNTextClassifier(vocab_size=vocab_size, embedding_dim=embedding_dim, output_dim=output_dim, kernel_sizes=kernel_sizes, num_filters=num_filters)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
metrics_callback = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=20, patience=3, min_delta=0.001)


In [ ]:
plot_metrics(metrics_callback)


In [ ]:
model.eval()
all_val_preds = []
all_val_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        all_val_preds.append(outputs)
        all_val_labels.append(labels)

In [ ]:
all_val_preds = torch.cat(all_val_preds)
all_val_labels = torch.cat(all_val_labels)

# Convertendo as previsões para rótulos
y_pred_val = torch.argmax(all_val_preds, dim=1).cpu().numpy()
y_true_val = all_val_labels.cpu().numpy()

# Calculando a matriz de confusão
cm = confusion_matrix(y_true_val, y_pred_val)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Matriz de Confusão')
plt.show()